# Objectness in few-shot video detection scores

Use a GPU for frozen encoder feature extraction and persistent storage for outputs. The temporal heads train on CPU. Both evaluation methods share the SAME three trained heads. The modification is `score = class_score * objectness ** beta` before the existing class-wise NMS. No additional training or trainable parameter is used for the proposed score.

This notebook prepares the original data and caches, trains the previously selected fixed reference, checks beta in {0, 0.5, 1} on validation, and stops unless the predeclared validation gate passes. If it passes, it prepares six untouched classes, fixes the selection, caches their features, records hashes, evaluates both methods, verifies AP, and exports uncertainty/results/shared weights. Earlier failed studies do not need to be rerun; their published manifests supply exclusion IDs. See `docs/SCORE_CALIBRATION_PLAN.md` for the exact protocol and limits. Do not choose settings from test results.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["USE_TF"] = "0"
BASE = (Path("/content") if Path("/content").is_dir() else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd())
REPO = BASE / "adaptive-fsvod"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/hskad/adaptive-fsvod.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Editable installation affects new Python processes. Also expose src to the
# notebook kernel, which was already running when pip created its .pth file.
PACKAGE_FILE = Path(subprocess.check_output(
    [sys.executable, "-c", "import adaptive_fsvod; print(adaptive_fsvod.__file__)"],
    text=True).strip()).resolve()
SOURCE_PATH = str(PACKAGE_FILE.parent.parent)
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
import importlib
importlib.invalidate_caches()
import adaptive_fsvod
print("Project imports:", adaptive_fsvod.__file__)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before feature extraction."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Set paths

For Colab persistence, mount Drive in a separate cell with `from google.colab import drive; drive.mount('/content/drive')`. Then set `EXPERIMENT_DIR`, for example, to `/content/drive/MyDrive/adaptive-fsvod-experiment`. Do not put experiment outputs in a read-only Kaggle input directory.


In [ ]:
EXPERIMENT_DIR = BASE / "adaptive-fsvod-experiment"  # Change to persistent storage.
IMAGE_ROOT = EXPERIMENT_DIR / "images"             # Selected frames download here.
SEED = 42
STEPS = 300
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def run(module, **flags):
    command = [sys.executable, "-m", "adaptive_fsvod." + module]
    for name, value in flags.items():
        if isinstance(value, bool):
            if value:
                command.append("--" + name.replace("_", "-"))
            continue
        command.append("--" + name.replace("_", "-"))
        if isinstance(value, (list, tuple)):
            command.extend(map(str, value))
        else:
            command.append(str(value))
    subprocess.run(command, check=True)


In [ ]:
RAW = EXPERIMENT_DIR / "raw"
RAW.mkdir(exist_ok=True)
ARCHIVE = RAW / "fsytv-annotations.zip"
BOXES = RAW / "fsytv-boxes.json"
if not ARCHIVE.exists():
    subprocess.run([sys.executable, "-m", "gdown", "1-m5Qcuxj4h8rryqnELtgnvOtTtWJ_Vys", "-O", str(ARCHIVE)], check=True)
if not BOXES.exists():
    run("convert_fsytv", archive=ARCHIVE, output=BOXES)

DATA = EXPERIMENT_DIR / "pilot"
run("prepare", annotation=BOXES, images_root=IMAGE_ROOT, output=DATA,
    class_counts=[6, 3, 3], max_videos_per_class=3, frame_stride=4, seed=SEED, skip_image_check=True)
run("download_frames", manifests=[DATA / f"{split}.json" for split in ["train", "val", "test"]],
    images_root=IMAGE_ROOT, range_cache=EXPERIMENT_DIR / "download_ranges")
from adaptive_fsvod.data import VideoManifest
for split in ["train", "val", "test"]:
    print(split, VideoManifest(DATA / f"{split}.json").validate_images())


## Cache frozen features

All methods share these caches. The encoder runs at its native resolution. Only the top 128 tokens are retained afterward. The cache can resume an interrupted extraction. Settings and annotation checksums must match.


In [ ]:
from adaptive_fsvod.cache import FrozenEncoder, build_cache
from adaptive_fsvod.data import VideoManifest, checksum, frame_key, read_json

CACHE = EXPERIMENT_DIR / "cache"
encoder = None
for split in ["train", "val", "test"]:
    manifest = VideoManifest(DATA / f"{split}.json")
    directory = CACHE / split
    metadata = read_json(directory / "metadata.json") if (directory / "metadata.json").exists() else {}
    expected = dict(manifest_sha256=checksum(manifest.path), encoder="google/owlv2-base-patch16-ensemble",
                    revision="cfd3195ba4ea9592eec887ded089f4c08eff231d", candidates=128,
                    support_per_class=16, pool_seed=42, synthetic=False)
    ready = metadata.get("complete") and all(metadata.get(k) == v for k, v in expected.items())
    if ready:
        pool = read_json(directory / "support_pool.json")
        needed = [directory / "frames" / f"{frame_key(vid, frame)}.pt"
                  for vid, video in manifest.videos.items() for frame in video["frames"]]
        needed += [directory / "support" / f"{item['key']}.pt" for values in pool.values() for item in values]
        ready = all(path.is_file() for path in needed)
    if ready:
        print(split, "complete cache reused without changing metadata")
    else:
        if encoder is None:
            encoder = FrozenEncoder(device="cuda", revision=expected["revision"], candidates=128)
        print(split, build_cache(manifest, directory, encoder, per_class=16))
if encoder is not None:
    del encoder
torch.cuda.empty_cache()


In [ ]:
for split, count, episode_seed in [("train", 300, 42), ("val", 20, 43), ("test", 30, 44)]:
    flags = dict(manifest=DATA / f"{split}.json", cache=CACHE / split,
                 output=DATA / f"{split}_episodes.json", episodes=count,
                 n_way=2, k_shot=1, seed=episode_seed)
    if split == "train":
        flags["max_frames"] = 4
    run("episodes", **flags)


## Train the fixed reference and run the bounded score study

The cell streams progress and reuses verified completed work. It stops on errors. Keep the storage mounted. A complete run can take time when new encoder features are needed. Bootstrap analysis uses query videos, not independent frames.


In [ ]:
import json

def run_script(script, *arguments):
    command = [sys.executable, "-u", str(REPO / "scripts" / script), *map(str, arguments)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)

RESULTS = EXPERIMENT_DIR / "exports" / "score_calibration_results"
PROTOCOL = RESULTS / "confirmation_protocol"
run_script("train_fixed_reference.py", "--experiment-dir", EXPERIMENT_DIR, "--threads", 4)
for stage in ["freeze", "validate", "export"]:
    run_script("run_score_calibration.py", "--experiment-dir", EXPERIMENT_DIR,
               "--stage", stage, "--output", RESULTS)
decision = json.loads((EXPERIMENT_DIR / "score_calibration" / "validation_results.json").read_text())
if decision["validation_gate_passed"]:
    for stage in ["prepare", "download", "cache", "freeze-cache"]:
        run_script("prepare_score_confirmation.py", "--experiment-dir", EXPERIMENT_DIR,
                   "--stage", stage, "--output", PROTOCOL,
                   "--device", "cuda" if torch.cuda.is_available() else "cpu", "--threads", 4)
    run_script("evaluate_score_confirmation.py", "--experiment-dir", EXPERIMENT_DIR, "--threads", 4)
    run_script("analyse_score_confirmation.py", "--experiment-dir", EXPERIMENT_DIR, "--output", RESULTS)
else:
    print("Validation gate failed. Stop this family. No new test data were scored.")
run_script("export_score_models.py", "--experiment-dir", EXPERIMENT_DIR)


In [ ]:
from IPython.display import display
from PIL import Image
comparison = RESULTS / "comparison.csv"
print(comparison.read_text() if comparison.exists() else (RESULTS / "validation.csv").read_text())
figure = RESULTS / "score_calibration.png"
if figure.exists():
    display(Image.open(figure))
print("Shared inference heads:", EXPERIMENT_DIR / "exports" / "score_calibration_inference.zip")
print("Use --objectness-beta 0 for baseline and --objectness-beta 1 for the selected score.")
